In [1]:
import typesense
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

True

In [2]:
import typesense

client = typesense.Client({
    'nodes': [{
        'host': 'p6ztcf5hybnxavmsp-1.a2.typesense.net',
        'port': '443',
        'protocol': 'https'
    }],
    'api_key': os.getenv('TYPESENSE_API_KEY'),
    'connection_timeout_seconds': 60
})

# Quick test to see if the key works
try:
    print("Testing connection...")
    print(client.collections.retrieve())
    print(" Connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")

Testing connection...
[{'created_at': 1790230801, 'curation_sets': [], 'default_sorting_field': '', 'enable_nested_fields': False, 'fields': [{'facet': False, 'hnsw_params': {'M': 16, 'ef_construction': 200}, 'index': True, 'infix': False, 'locale': '', 'name': 'vec', 'num_dim': 768, 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'float[]', 'vec_dist': 'cosine'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'text', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': '.*', 'optional': True, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'auto'}], 'name': 'lang-chain', 'num_documents': 11, 'symbols_to_index': [], 'synonym_sets': [], 'token_separators': []}, {'created_at': 1790229722, 'curation_sets': [],

In [3]:
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [4]:
client

In [5]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [7]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [ ]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\nathm\AppData\Local\Temp\ipykernel_19812\2070461199.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\nathm\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

In [10]:
source_candidates = [Path('example.txt'), Path('Typesense/example.txt')]
source_path = next((path for path in source_candidates if path.exists()), None)
if source_path is None:
    raise FileNotFoundError('Could not find Typesense/example.txt')

loader = TextLoader(str(source_path), encoding='utf-8')
documents= loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 620.44it/s]


In [15]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "p6ztcf5hybnxavmsp-1.a2.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": os.getenv('TYPESENSE_API_KEY'),
        "typesense_collection_name": "lang-chain"
    },
    
)

In [16]:
google_api_key = os.getenv('GOOGLE_API_KEY')
if not google_api_key:
    raise RuntimeError('GOOGLE_API_KEY is missing. Add it to the project .env file.')

llm = ChatGoogleGenerativeAI(
    model=os.getenv('GOOGLE_MODEL', 'gemini-3.6-flash'),
    temperature=0,
)

def ask_typesense(question: str) -> str:
    retrieved_docs = docsearch.similarity_search(question, k=4)
    context = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    prompt = f'''Answer the question using only the context below.
If the answer is not in the context, say that you do not know.

Context:
{context}

Question: {question}
Answer:'''
    response = llm.invoke(prompt)
    return response.content


In [17]:
print(ask_typesense("what  is lstm"))

c:\Users\nathm\Desktop\RAG\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Based on the provided context, a Long Short-Term Memory (LSTM) network is an upgrade to regular Recurrent Neural Networks (RNNs) used to solve sequence-based problems without struggling with short-term memory issues. \n\nIts primary focus is to discard as much unnecessary information as possible, which it achieves using three sections (the forget section, the input section, and the output section) and "gates" (neurons with a sigmoid activation function) that decide what proportion of information to retain.', 'extras': {'signature': 'EukQCuYQAWkUfRNAJnkvtpm0pxaVj5HaeBPYQhYsVYUBb4np2uaBSj9fXAh15NEl5Tj59/HP9qsDksiTxS6oS1EGsQFp8So0L025W5BZrPk1448l2fEVmbRfXJSLpBsGO4VlPosQF/jgSoeGRsOyJph9AQFz8BJCeg/60/KnosnWeMfCdoyRrPQgcju5LYIoS0OaP2JAh3bNUBgtkQsNqPEfHcdIbZtU1tWexEbTJiL182S5ZtwUwjrQlwW74orCVridEkowzmfrtZTC19kXBkvHb/O5Ov8HAUThjz/mowWU4Q4NO3YoJCKOrd0IcqL1MsYmnjj8ntC9PF0tda8ksobUM3IqByGopdKzF9J6PcA2OmzlfF38atjnWRTc0wpZEUHN9FaHvutxoxZVmkhR/UVi7R0d7pZWLtpKSzE+2Vj/aSW/yp